# Homework 8 · Exercise 2
## Fisher analysis of GW150914 with Hanford strain

Fit the detector-frame masses of GW150914 using a zero-spin TaylorF2 inspiral
waveform. Compare the exact likelihood surface on your mass grid with its
local Fisher approximation.

Run the setup cells, complete the three coding tasks, and follow the
submission instructions in the handout.
**Download your completed `.ipynb` before closing Binder.** Binder sessions
are temporary.

To export a PDF, run the cells and save your notebook in Binder. In JupyterLab, open **File**, choose **Save and Export Notebook As**, then select **PDF via LaTeX**.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from hw8_setup import component_masses, load_spectrum
from taylorf2 import SOLAR_MASS_SECONDS, taylorf2

## Strain and templates

Use the GW150914-v3 strain from the LIGO Hanford detector. Estimate the
one-sided noise power spectral density (PSD) from the first 12 seconds of
the file using 2-second Welch segments. The 4-second on-source spectrum
uses a Tukey window and a Fourier time origin of GPS 1126259462.

TaylorF2 evaluates the strain spectrum directly on the same positive-frequency
bins, from 30 to 46 Hz inclusive. Both spectra use the
$e^{-2\pi i f t}$ Fourier convention and have units of strain × seconds.
The upper frequency lies below the Schwarzschild innermost stable circular
orbit (ISCO) frequency across the grid. The fixed band keeps the bins in the
likelihood sum identical for every mass pair.

The setup fixes the time offset, detector amplitude scale, and Fourier phase
rotation using a preliminary fit. The model's intrinsic waveform amplitude
still depends on the masses.

In [ ]:
data_dir = Path("data")
strain_path = data_dir / "H-H1_GWOSC_4KHZ_R1-1126259447-32.hdf5"
inspiral_band = (30, 46)
frequencies, strain, psd, df = load_spectrum(strain_path, inspiral_band)

time_offset = 0.387507505683
amplitude_scale = 0.398852643467
fourier_phase = 2.815001078781
hanford_factor = amplitude_scale * np.exp(
    1j * fourier_phase - 2j * np.pi * frequencies * time_offset
)


def hanford_waveform(chirp_mass, eta):
    return hanford_factor * taylorf2(frequencies, chirp_mass, eta)

## Mass grid

Work in chirp mass $\mathcal M=M\eta^{3/5}$ and symmetric mass ratio
$\eta=m_1m_2/M^2$, where $M=m_1+m_2$ and $m_1\ge m_2$. The grid
includes the equal-mass endpoint $\eta=1/4$. Mass inputs and plotted
detector-frame component masses are in solar masses.

In [ ]:
chirp_masses = np.linspace(24.0, 35.0, 441)
etas = np.linspace(0.190, 0.250, 121)
chirp_grid, eta_grid = np.meshgrid(chirp_masses, etas, indexing="xy")
m1_grid, m2_grid = component_masses(chirp_grid, eta_grid)

largest_total_mass = (chirp_masses[-1] + 0.01) * (etas[0] - 0.0005)**(-3 / 5)
min_f_isco = 1 / (6**1.5 * np.pi * largest_total_mass * SOLAR_MASS_SECONDS)
assert frequencies[-1] < min_f_isco  # includes the derivative steps

example_template = hanford_waveform(30.0, 0.24)
assert example_template.shape == strain.shape == psd.shape
print(f"{frequencies.size} frequency bins at {df:g} Hz spacing")
print(f"{chirp_grid.size:,} mass pairs including equal masses")

## Task 1 · Evaluate the log likelihood ratio

For each mass pair, generate a template $\tilde h$ and use the one-sided PSD
in the Gaussian log likelihood ratio
$$
(a\mid b)=4\,\Delta f\,\operatorname{Re}\sum_k
\frac{\tilde a_k\tilde b_k^*}{S_n(f_k)}, \qquad
\ln\Lambda=(d\mid h)-\tfrac12(h\mid h).
$$

Implement the inner product and fill a 2D array `log_likelihood` of the
same shape as `chirp_grid`. Its rows index `etas` and its columns index
`chirp_masses`. Include the last row. Save the maximizing array index as
the tuple `best_index` and convert the corresponding grid point to
$(m_1,m_2)$ to report the best-fit masses.

In [ ]:
# TODO 1: Implement the inner product, evaluate the grid, and report the best fit.
log_likelihood = None
best_index = None

## Task 2 · Compute the Fisher matrix

At the best-fitting grid point, use centered differences in both mass
coordinates, holding the other coordinate fixed. Use steps of
$0.01\,M_\odot$ in $\mathcal M$ and $0.0005$ in $\eta$.

At equal masses, the upper stencil point is above $\eta=1/4$. Evaluate
it with `hanford_waveform`, which uses a TaylorF2 formula that extends
smoothly across the boundary. Use that point only for the derivative.
The likelihood grid and the conversion to component masses still use
$\eta\leq1/4$.

Assemble $\Gamma_{ij}=(\partial_i h\mid\partial_j h)$ in the order
$(\mathcal M,\eta)$. Use the same inner product as in Task 1 and save your
matrix as `fisher`.

In [ ]:
# TODO 2: Compute the waveform derivatives and the 2 × 2 Fisher matrix.
fisher = None

## Task 3 · Overlay the mass contours

Evaluate
$$
Q(\theta)=(\theta-\hat\theta)^{\mathsf T}\Gamma(\theta-\hat\theta),
\qquad \theta=(\mathcal M,\eta),
$$
on the same grid and save it as `fisher_quadratic`. The plotting cell
overlays contours of $Q$ and $2(\ln\Lambda_{\max}-\ln\Lambda)$ at 1.39
and 4.61, with $m_1$ horizontal and $m_2$ vertical.

The levels 1.39 and 4.61 are the 50% and 90% quantiles of $\chi^2_2$.
For an unrestricted two-dimensional Gaussian, the probability within a
contour of height $Q$ is $1-e^{-Q/2}$. Here the equal-mass boundary and
deviations from a Gaussian can change the enclosed probabilities.

Compare the tilt and equal-mass edge with the lecture mass plot. Figure 1
shows source-frame Bayesian credible regions. Your plot shows detector-frame
contours from Hanford with time, amplitude, and phase held fixed.
[GW150914 properties paper, Figure 1](https://arxiv.org/pdf/1602.03840#page=6)


In [ ]:
# TODO 3: Evaluate the Fisher quadratic at every grid point.
fisher_quadratic = None

In [ ]:
levels = (1.39, 4.61)


def draw_contours(ax, values, color, style, label):
    ax.contour(m1_grid, m2_grid, values, levels=levels,
               colors=[color], linestyles=style, linewidths=1.5)
    return Line2D([], [], color=color, linestyle=style, label=label)


def mass_axes(ax):
    equal_mass, = ax.plot(m1_grid[-1], m2_grid[-1], color="0.4", linestyle=":",
                         label="Equal masses")
    search_edge, = ax.plot(m1_grid[0], m2_grid[0], color="0.55", linestyle="-.",
                          label=r"Grid edge ($\eta=0.19$)")
    ax.set(xlabel=r"Detector-frame $m_1$ [$M_\odot$]",
           ylabel=r"Detector-frame $m_2$ [$M_\odot$]",
           xlim=(27, 69), ylim=(15, 42), aspect="equal")
    return equal_mass, search_edge


if all(value is not None for value in (log_likelihood, best_index, fisher_quadratic)):
    likelihood_drop = 2 * (np.max(log_likelihood) - log_likelihood)
    fig, ax = plt.subplots(figsize=(8, 5.5), layout="constrained")
    likelihood_line = draw_contours(ax, likelihood_drop, "tab:blue", "-", "TaylorF2 likelihood")
    fisher_line = draw_contours(ax, fisher_quadratic, "tab:orange", "--", "Fisher quadratic")
    best, = ax.plot(m1_grid[best_index], m2_grid[best_index], "o", color="0.2",
                    markersize=4, label="Best fit")
    ax.legend(handles=[likelihood_line, fisher_line, best, *mass_axes(ax)])
    plt.show()
else:
    print("Complete Tasks 1–3 to display the likelihood and Fisher overlay.")